# 06 · Embeddings with Sentence Transformers
**Goal:** generate real text embeddings and use them for a small semantic search workflow.

The model below is a public Sentence Transformers model. The first run may download model files and therefore needs internet access.

In [ ]:
%pip install -q sentence-transformers scikit-learn pandas

## Load the embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model ready")

## Embed a mini knowledge base

In [ ]:
knowledge = [
    {"id": "python_basics", "text": "Python uses indentation to define code blocks and supports many programming styles."},
    {"id": "model_training", "text": "Machine learning training adjusts model parameters using examples and an objective function."},
    {"id": "vector_search", "text": "Semantic search retrieves passages whose vector representations are close to a query embedding."},
    {"id": "nutrition", "text": "A balanced diet includes appropriate energy, protein, fats, carbohydrates, vitamins, and minerals."},
    {"id": "database", "text": "A relational database stores structured information in tables linked by keys."},
]
texts = [item["text"] for item in knowledge]
vectors = model.encode(texts, normalize_embeddings=True)
print("Embedding matrix shape:", vectors.shape)

## Search by meaning

In [ ]:
def search(query, top_k=3):
    query_vector = model.encode([query], normalize_embeddings=True)
    scores = (query_vector @ vectors.T)[0]
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {"score": float(scores[i]), **knowledge[i]}
        for i in order
    ]

for hit in search("How does retrieval find related passages?", top_k=3):
    print(f"{hit['score']:.3f} | {hit['id']} | {hit['text']}")

## Inspect similarity matrix

In [ ]:
matrix = cosine_similarity(vectors)
labels = [item["id"] for item in knowledge]
pd.DataFrame(matrix, index=labels, columns=labels).round(2)

## Optional: visualize embeddings
PCA is useful for a rough view of high-dimensional vectors, but a 2D plot can hide important structure. It should not be treated as a quality metric.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

points = PCA(n_components=2).fit_transform(vectors)
plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1])
for i, label in enumerate(labels):
    plt.annotate(label, (points[i, 0], points[i, 1]), xytext=(4, 4),
                 textcoords="offset points")
plt.title("Mini knowledge base embeddings (PCA projection)")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.tight_layout()
plt.show()

### Next steps
- Replace the sample records with content from a domain you care about.
- Compare `normalize_embeddings=True` with unnormalized vectors.
- Add a relevance threshold only after evaluating examples.
- Combine this retrieval step with an LLM and cite the retrieved source passages to build a grounded RAG assistant.